# RealWinner EA v5 — Backtest su Dati REALI Dukascopy
**Premi Runtime → Run all — scarica automaticamente dati reali EURUSD M15**

- Dati reali da Dukascopy (fonte istituzionale, usata da banche)
- Jan 2024 → oggi, scaricati automaticamente
- Nessun file da caricare manualmente


In [ ]:
!pip install duka --quiet
!pip install numpy --quiet
print('OK')


## Download dati reali EURUSD M15 da Dukascopy

In [ ]:
import subprocess, os
from datetime import datetime, timedelta

# Scarica EURUSD M15 da Dukascopy
# Dukascopy e' la fonte usata da banche e istituzionali — dati reali
print('Scaricando dati reali EURUSD M15 da Dukascopy...')
print('(potrebbe richiedere 1-2 minuti)')

result = subprocess.run(
    ['duka', 'EURUSD', '-s', '2024-01-01', '-e', '2025-12-31', '-c', 'M15', '-f', '/content/', '--header'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'nessun output')
if result.returncode != 0:
    print('Stderr:', result.stderr[-300:])

# Trova il file scaricato
import glob
files = glob.glob('/content/EURUSD*.csv') + glob.glob('/content/*.csv')
print('File trovati:', files)


## Parsing dati reali

In [ ]:
import csv
from datetime import datetime

def load_dukascopy_csv(filepath):
    bars = []
    with open(filepath, newline='') as f:
        reader = csv.reader(f)
        for row in reader:
            if not row or len(row) < 5: continue
            # Skip header
            if any(h in row[0].lower() for h in ['date','time','gmt','local']): continue
            try:
                # Dukascopy format: datetime, open, high, low, close, volume
                dt_str = row[0].strip()
                # Try different formats
                for fmt in ['%d.%m.%Y %H:%M:%S.%f', '%Y.%m.%d %H:%M', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M']:
                    try:
                        dt = datetime.strptime(dt_str[:19], fmt[:len(dt_str[:19])])
                        break
                    except: continue
                else: continue
                wd = dt.weekday()
                if wd >= 5: continue
                bars.append({
                    'dt': dt, 'open': float(row[1]), 'high': float(row[2]),
                    'low': float(row[3]), 'close': float(row[4]),
                    'volume': int(float(row[5])) if len(row) > 5 else 1000,
                    'hour': dt.hour, 'weekday': wd
                })
            except Exception as e:
                continue
    return sorted(bars, key=lambda x: x['dt'])

# Carica il file
if files:
    REAL_BARS = load_dukascopy_csv(files[0])
    print(f'Caricate {len(REAL_BARS):,} barre reali')
    if REAL_BARS:
        import statistics
        atrs = [REAL_BARS[i]['high']-REAL_BARS[i]['low'] for i in range(1,min(1000,len(REAL_BARS)))]
        print(f'Da: {REAL_BARS[0]["dt"]}')
        print(f'A:  {REAL_BARS[-1]["dt"]}')
        print(f'ATR medio M15: {round(statistics.mean(atrs)*10000,1)} pips')
        print(f'Prezzo range: {min(b["close"] for b in REAL_BARS):.5f} - {max(b["close"] for b in REAL_BARS):.5f}')
else:
    print('Nessun file trovato - uso dati sintetici')
    REAL_BARS = None


## Engine completo v5

In [ ]:
"""
RealWinner EA v1.0 — Python Backtest Engine
Faithful port of all MQL5 strategy logic:
  - SMC: Order Blocks + FVG + BOS + HTF Bias
  - Trend: EMA 9/21/50 + RSI + Volume
  - Mean Reversion: BB + RSI (ranging only)
  - Confluence: 2-of-3 required
  - Full risk management: partial close, BE, trailing, daily/total DD guards
"""

import numpy as np
import random
from datetime import datetime, timedelta

# ─────────────────────────────────────────────────────────────────────────────
# PARAMETERS (matching MQL5 defaults)
# ─────────────────────────────────────────────────────────────────────────────
P = dict(
    use_smc=True, use_trend=True, use_mr=True, require_confluence=True,
    confluence_min=2,
    london=(7, 11), newyork=(13, 18), overlap=(11, 13),
    use_overlap=True, asia=(0, 4),
    avoid_news=True, close_weekend=True,
    risk_pct=0.9,
    max_daily_loss=2.4,
    daily_loss_warning=0.7,
    max_total_dd=5.5,
    max_trades_day=10,
    max_consec_losses=3,
    scale_out=True, use_be=True, use_trail=True,
    be_trigger_rr=1.0,
    tp1_rr=1.5, tp2_rr=3.0, min_rr=1.1, max_spread_pts=28,
    ob_lookback=50, ob_strength=2, ob_body_min=0.00010,
    use_fvg=False, fvg_lookback=20,
    ema_fast=9, ema_med=21, ema_slow=50, ema_200=200,
    rsi_period=14, rsi_long_min=50, rsi_long_max=78,
    rsi_short_min=22, rsi_short_max=50,
    atr_period=14, atr_sl_mult=1.1, use_vol=False,
    bb_period=20, bb_dev=2.0, mr_ob=70, mr_os=30,
    start_balance=10000.0,
)

# ─────────────────────────────────────────────────────────────────────────────
# DATA GENERATOR — realistic EURUSD M15, Jan 2024 → Mar 2026
# ─────────────────────────────────────────────────────────────────────────────
def generate_eurusd_m15(start="2024-01-02", days=450, seed=42):
    """
    Generates realistic EURUSD M15 OHLCV with:
    - Regime shifts (trending / ranging)
    - Intraday volatility profiles (London/NY spikes)
    - Realistic spread and volume
    - Weekend gaps
    - Macro shock events
    """
    rng = np.random.default_rng(seed)
    bars_per_day = 24 * 4  # M15
    total_bars = days * bars_per_day

    bars = []
    price = 1.08200
    dt = datetime.strptime(start, "%Y-%m-%d")

    # Regime state
    regime = "ranging"   # "bull", "bear", "ranging"
    regime_countdown = rng.integers(80, 200)
    trend_bias = 0.0

    # Macro shock schedule (approximate real events)
    shock_dates = {
        "2024-03-20", "2024-06-12", "2024-09-18",
        "2024-11-07", "2025-01-29", "2025-03-19",
        "2025-06-11", "2025-09-17", "2025-12-10",
        "2024-02-02", "2024-05-03", "2024-08-02",  # NFP dates
        "2025-02-07", "2025-05-02", "2025-08-01",
    }

    vol_series = []

    for i in range(total_bars):
        weekday = dt.weekday()  # 0=Mon
        hour = dt.hour
        minute = dt.minute

        # Skip weekends
        if weekday >= 5:
            dt += timedelta(minutes=15)
            continue

        # ── Regime shift
        regime_countdown -= 1
        if regime_countdown <= 0:
            prev = regime
            regime = rng.choice(["bull", "bear", "ranging"], p=[0.35, 0.30, 0.35])
            regime_countdown = int(rng.integers(60, 250))
            if regime == "bull":   trend_bias =  0.00004
            elif regime == "bear": trend_bias = -0.00004
            else:                  trend_bias =  0.0

        # ── Intraday volatility multiplier
        if 7 <= hour < 9:     vol_mult = 1.4   # London open
        elif 9 <= hour < 11:  vol_mult = 1.6   # London peak
        elif 12 <= hour < 13: vol_mult = 0.7   # Lunch
        elif 13 <= hour < 15: vol_mult = 1.8   # NY open
        elif 15 <= hour < 17: vol_mult = 1.5   # NY peak
        elif 20 <= hour or hour < 2: vol_mult = 0.4  # Overnight
        else:                 vol_mult = 0.9

        # ── Macro shock
        date_str = dt.strftime("%Y-%m-%d")
        shock = 3.5 if date_str in shock_dates and 13 <= hour <= 14 else 1.0

        # ── Base move
        base_vol = 0.00008 * vol_mult * shock
        drift    = trend_bias * vol_mult
        move     = rng.normal(drift, base_vol)

        # ── OHLC construction
        o = price
        body = rng.normal(move, base_vol * 0.4)
        c = o + body
        wick_up   = abs(rng.normal(0, base_vol * 0.6))
        wick_down = abs(rng.normal(0, base_vol * 0.6))
        h = max(o, c) + wick_up
        l = min(o, c) - wick_down

        # ── Clamp to realistic range
        price = np.clip(c, 1.0400, 1.1600)
        h = max(h, price); l = min(l, price)

        # ── Volume (correlated with volatility)
        vol = int(rng.normal(1200 * vol_mult * shock, 300))
        vol = max(100, vol)
        vol_series.append(vol)

        bars.append({
            "dt": dt, "open": round(o, 5), "high": round(h, 5),
            "low": round(l, 5), "close": round(price, 5), "volume": vol,
            "hour": hour, "weekday": weekday,
        })
        dt += timedelta(minutes=15)

    return bars

# ─────────────────────────────────────────────────────────────────────────────
# INDICATOR CALCULATIONS
# ─────────────────────────────────────────────────────────────────────────────
def ema(series, period):
    k = 2.0 / (period + 1)
    result = [series[0]]
    for v in series[1:]:
        result.append(v * k + result[-1] * (1 - k))
    return result

def rsi(closes, period=14):
    gains, losses = [], []
    for i in range(1, len(closes)):
        d = closes[i] - closes[i-1]
        gains.append(max(d, 0)); losses.append(max(-d, 0))
    result = [50.0] * len(closes)
    if len(gains) < period:
        return result
    avg_g = np.mean(gains[:period])
    avg_l = np.mean(losses[:period])
    for i in range(period, len(gains)):
        avg_g = (avg_g * (period-1) + gains[i]) / period
        avg_l = (avg_l * (period-1) + losses[i]) / period
        rs = avg_g / avg_l if avg_l > 0 else 100
        result[i+1] = 100 - 100 / (1 + rs)
    return result

def atr(bars, period=14):
    tr_list = [abs(bars[0]["high"] - bars[0]["low"])]
    for i in range(1, len(bars)):
        hi, lo, pc = bars[i]["high"], bars[i]["low"], bars[i-1]["close"]
        tr_list.append(max(hi-lo, abs(hi-pc), abs(lo-pc)))
    result = [tr_list[0]]
    for i in range(1, len(tr_list)):
        result.append((result[-1] * (period-1) + tr_list[i]) / period)
    return result

def bollinger(closes, period=20, dev=2.0):
    mid, upper, lower = [], [], []
    for i in range(len(closes)):
        if i < period - 1:
            mid.append(closes[i]); upper.append(closes[i]); lower.append(closes[i])
        else:
            sl = closes[i-period+1:i+1]
            m  = np.mean(sl)
            s  = np.std(sl, ddof=0)
            mid.append(m); upper.append(m + dev*s); lower.append(m - dev*s)
    return mid, upper, lower

# H4 EMA — resample M15 into H4 (every 16 bars)
def compute_h4_emas(bars, fast=9, slow=50, ema200=200):
    h4_closes = []
    for i in range(0, len(bars), 16):
        chunk = bars[i:i+16]
        if chunk:
            h4_closes.append(chunk[-1]["close"])
    if not h4_closes:
        return {}, {}, {}
    h4_ema_fast = ema(h4_closes, fast)
    h4_ema_slow = ema(h4_closes, slow)
    h4_ema_200  = ema(h4_closes, ema200)
    # Map back to M15 index
    fast_map, slow_map, e200_map = {}, {}, {}
    for idx, i in enumerate(range(0, len(bars), 16)):
        for j in range(i, min(i+16, len(bars))):
            fast_map[j] = h4_ema_fast[idx]
            slow_map[j] = h4_ema_slow[idx]
            e200_map[j] = h4_ema_200[idx]
    return fast_map, slow_map, e200_map

# ─────────────────────────────────────────────────────────────────────────────
# SMC LOGIC
# ─────────────────────────────────────────────────────────────────────────────
def scan_order_blocks(bars, i, ob_lookback, ob_strength, ob_body_min):
    bull_obs, bear_obs = [], []
    start = max(0, i - ob_lookback)
    for k in range(start, max(0, i - ob_strength - 1)):
        o = bars[k]["open"]; c = bars[k]["close"]
        h = bars[k]["high"]; l = bars[k]["low"]
        body = abs(c - o)
        if body < ob_body_min:
            continue
        # Bullish OB: bearish candle + impulse up after
        if c < o:
            valid_js = [j for j in range(1, ob_strength+1) if k+j < i]
            if len(valid_js) == ob_strength:
                impulse = all(bars[k+j]["close"] > bars[k+j-1]["close"] for j in valid_js)
                if impulse:
                    bull_obs.append({"high": h, "low": l, "mid": (h+l)/2})
        # Bearish OB: bullish candle + impulse down after
        if c > o:
            valid_js = [j for j in range(1, ob_strength+1) if k+j < i]
            if len(valid_js) == ob_strength:
                impulse = all(bars[k+j]["close"] < bars[k+j-1]["close"] for j in valid_js)
                if impulse:
                    bear_obs.append({"high": h, "low": l, "mid": (h+l)/2})
    return bull_obs[-3:], bear_obs[-3:]  # keep most recent

def scan_fvg(bars, i, fvg_lookback):
    bull_fvg, bear_fvg = [], []
    start = max(2, i - fvg_lookback)
    for k in range(start, i-1):
        h1 = bars[k-1]["high"]; l1 = bars[k-1]["low"]
        h3 = bars[k+1]["high"] if k+1 < len(bars) else bars[k]["high"]
        l3 = bars[k+1]["low"]  if k+1 < len(bars) else bars[k]["low"]
        if l3 > h1:
            bull_fvg.append({"high": l3, "low": h1})
        if h3 < l1:
            bear_fvg.append({"high": l1, "low": h3})
    return bull_fvg[-2:], bear_fvg[-2:]

def get_smc_signal(bars, i, closes, htf_bull_bias, h4_fast, h4_slow, h4_200, p):
    if i < p["ob_lookback"] + 5:
        return 0
    bid = bars[i]["close"]

    # HTF Bias
    if i in h4_200:
        htf_bull = (bid > h4_200[i]) and (h4_fast.get(i,0) > h4_slow.get(i,0))
    else:
        htf_bull = htf_bull_bias

    bull_obs, bear_obs = scan_order_blocks(bars, i, p["ob_lookback"], p["ob_strength"], p["ob_body_min"])
    bull_fvg, bear_fvg = scan_fvg(bars, i, p["fvg_lookback"]) if p["use_fvg"] else ([], [])

    # BOS check
    lookback = min(30, i)
    sw_high = max(bars[j]["high"] for j in range(i-lookback, i))
    sw_low  = min(bars[j]["low"]  for j in range(i-lookback, i))
    bull_bos = bars[i]["close"] > sw_high
    bear_bos = bars[i]["close"] < sw_low

    if htf_bull or bull_bos:
        for ob in bull_obs:
            if ob["low"] <= bid <= ob["high"]:
                if not p["use_fvg"]:
                    return 1
                for fvg in bull_fvg:
                    if abs(bid - fvg["low"]) < 0.0020:
                        return 1
                if not bull_fvg:
                    return 1

    if not htf_bull or bear_bos:
        for ob in bear_obs:
            if ob["low"] <= bid <= ob["high"]:
                if not p["use_fvg"]:
                    return -1
                for fvg in bear_fvg:
                    if abs(bid - fvg["high"]) < 0.0020:
                        return -1
                if not bear_fvg:
                    return -1
    return 0

def get_trend_signal(i, closes, vols, ema_f, ema_m, ema_s, rsi_v, atr_v, h4_fast, h4_slow, h4_200, p):
    if i < max(p["ema_slow"], p["rsi_period"]) + 5:
        return 0

    ef = ema_f[i]; em = ema_m[i]; es = ema_s[i]; rs = rsi_v[i]

    # EMA alignment
    bull_align = (ef > em > es)
    bear_align = (ef < em < es)

    # EMA crossover (M15 i-1 vs i)
    bull_cross = (ema_f[i] > ema_m[i]) and (ema_f[i-1] <= ema_m[i-1])
    bear_cross = (ema_f[i] < ema_m[i]) and (ema_f[i-1] >= ema_m[i-1])

    # RSI
    rsi_long  = p["rsi_long_min"]  <= rs <= p["rsi_long_max"]
    rsi_short = p["rsi_short_min"] <= rs <= p["rsi_short_max"]

    # Volume
    vol_ok = True
    if p["use_vol"] and i >= 20:
        avg_vol = np.mean(vols[i-20:i])
        vol_ok  = (vols[i] >= avg_vol * 1.3)

    # HTF alignment
    htf_bull = closes[i] > h4_200.get(i, closes[i]) and h4_fast.get(i,0) > h4_slow.get(i,0)
    htf_bear = closes[i] < h4_200.get(i, closes[i]) and h4_fast.get(i,0) < h4_slow.get(i,0)

    if bull_align and (bull_cross or bull_align) and rsi_long  and vol_ok and (htf_bull or bull_align):
        return 1
    if bear_align and (bear_cross or bear_align) and rsi_short and vol_ok and (htf_bear or bear_align):
        return -1
    return 0

def get_mr_signal(i, closes, ema_f, ema_s, rsi_v, bb_upper, bb_lower, p):
    if i < max(p["bb_period"], 3):
        return 0
    # Only in ranging (EMA spread small)
    ranging = abs(ema_f[i] - ema_s[i]) < 0.0020
    if not ranging:
        return 0
    c  = closes[i]; cp = closes[i-1]
    rs = rsi_v[i]
    if cp <= bb_lower[i-1] and c > bb_lower[i] and rs <= p["mr_os"] + 5:
        return 1
    if cp >= bb_upper[i-1] and c < bb_upper[i] and rs >= p["mr_ob"] - 5:
        return -1
    return 0

# ─────────────────────────────────────────────────────────────────────────────
# MAIN BACKTEST
# ─────────────────────────────────────────────────────────────────────────────
def run_backtest(bars, p):
    n = len(bars)
    closes = [b["close"] for b in bars]
    highs  = [b["high"]  for b in bars]
    lows   = [b["low"]   for b in bars]
    vols   = [b["volume"]for b in bars]

    # Pre-compute indicators
    ema_f   = ema(closes, p["ema_fast"])
    ema_m   = ema(closes, p["ema_med"])
    ema_s   = ema(closes, p["ema_slow"])
    rsi_v   = rsi(closes, p["rsi_period"])
    atr_v   = atr(bars,   p["atr_period"])
    bb_mid, bb_upper, bb_lower = bollinger(closes, p["bb_period"], p["bb_dev"])
    h4_fast, h4_slow, h4_200   = compute_h4_emas(bars, p["ema_fast"], p["ema_slow"], p["ema_200"])

    # State
    balance      = p["start_balance"]
    equity       = balance
    peak_equity  = balance
    day_start_bal= balance
    prev_date    = bars[0]["dt"].date()

    trades       = []
    open_trade   = None
    trades_today = 0
    consec_loss  = 0
    daily_limit  = False
    total_dd_hit = False

    equity_curve  = []
    dd_curve      = []
    htf_bull_bias = True
    daily_warning = False
    extra_trades  = 0
    prev_month      = bars[0]["dt"].strftime("%Y-%m")
    month_start_bal = balance
    monthly_halved  = False

    # ─── BAR LOOP ─────────────────────────────────────────────────────────────
    for i in range(max(p["ob_lookback"], p["ema_200"])+10, n):
        b    = bars[i]
        date = b["dt"].date()
        hour = b["hour"]
        wday = b["weekday"]

        # ── Daily reset + monthly circuit breaker
        if date != prev_date:
            # Monthly circuit breaker: if monthly loss > 3%, reduce risk 50% rest of month
            cur_month = date.strftime("%Y-%m")
            if cur_month != prev_month:
                month_start_bal = balance
                prev_month = cur_month
                monthly_halved = False
            month_loss_pct = (month_start_bal - balance) / month_start_bal * 100
            if month_loss_pct >= 2.5 and not monthly_halved:
                monthly_halved = True  # flag to halve risk rest of month

            day_start_bal  = balance
            trades_today   = 0
            consec_loss    = 0
            daily_limit    = False
            daily_warning  = False
            extra_trades   = 0
            prev_date      = date

        # ── Equity update — use worst intrabar price for accurate DD
        if open_trade:
            ot = open_trade
            if ot["dir"] == 1:
                # Worst case for longs = bar low
                worst = (lows[i] - ot["entry"]) * 100000 * ot["lots"]
                best  = (highs[i] - ot["entry"]) * 100000 * ot["lots"]
                ot["unrealised"] = (closes[i] - ot["entry"]) * 100000 * ot["lots"]
            else:
                worst = (ot["entry"] - highs[i]) * 100000 * ot["lots"]
                best  = (ot["entry"] - lows[i])  * 100000 * ot["lots"]
                ot["unrealised"] = (ot["entry"] - closes[i]) * 100000 * ot["lots"]
            equity = balance + ot["unrealised"]
            # Track worst intrabar equity for accurate DD
            worst_eq = balance + worst
        else:
            equity = balance
            worst_eq = equity

        if equity > peak_equity:
            peak_equity = equity

        dd_pct       = (peak_equity - equity) / peak_equity * 100
        daily_dd_pct = (day_start_bal - equity) / day_start_bal * 100

        equity_curve.append(equity)
        dd_curve.append(dd_pct)

        # ── Hard stops
        if dd_pct >= p["max_total_dd"]:
            if open_trade:
                pnl = open_trade["unrealised"]
                balance += pnl
                open_trade["exit"] = closes[i]
                open_trade["pnl"]  = round(pnl, 2)
                open_trade["exit_reason"] = "TOTAL_DD"
                trades.append(open_trade)
                open_trade = None
            total_dd_hit = True
            break

        # Daily loss warning at 1% — restrict to 1 more trade max
        warn_thresh = p.get("daily_loss_warning", 1.0)
        if daily_dd_pct >= warn_thresh and not daily_warning:
            daily_warning = True

        # Hard daily stop at max_daily_loss%
        if daily_dd_pct >= p["max_daily_loss"]:
            if open_trade:
                pnl = open_trade["unrealised"]
                balance += pnl
                open_trade["exit"] = closes[i]
                open_trade["pnl"]  = round(pnl, 2)
                open_trade["exit_reason"] = "DAILY_DD"
                trades.append(open_trade)
                open_trade = None
            daily_limit = True

        # ── Manage open trade (BE, trail, partial, TP/SL)
        if open_trade:
            ot = open_trade
            c  = closes[i]; h = highs[i]; lo = lows[i]
            atr_cur = atr_v[i]

            sl_dist_orig = abs(ot["entry"] - ot["sl"])

            # ── Break-Even: trigger at be_trigger_rr (default R:R 1.0)
            be_rr = p.get("be_trigger_rr", 1.0)
            be_target = ot["entry"] + sl_dist_orig * be_rr * ot["dir"]
            be_hit = (ot["dir"]==1 and h >= be_target) or (ot["dir"]==-1 and lo <= be_target)
            if p["use_be"] and be_hit and not ot.get("be_done"):
                new_be_sl = ot["entry"] + (8e-5 * ot["dir"])  # BE + tiny buffer
                if ot["dir"] == 1 and new_be_sl > ot["sl"]:
                    ot["sl"] = new_be_sl; ot["be_done"] = True
                if ot["dir"] == -1 and new_be_sl < ot["sl"]:
                    ot["sl"] = new_be_sl; ot["be_done"] = True

            # ── Check TP1 for partial close
            if not ot.get("tp1_done"):
                tp1_hit = (ot["dir"]==1 and h >= ot["tp1"]) or (ot["dir"]==-1 and lo <= ot["tp1"])
                if tp1_hit and p["scale_out"]:
                    partial_pnl = (ot["tp1"] - ot["entry"]) * 100000 * ot["lots"] * 0.5 * ot["dir"]
                    balance    += partial_pnl
                    ot["lots"] *= 0.5
                    ot["tp1_done"] = True
                    ot["partial_pnl"] = round(partial_pnl, 2)

            # ── Trailing stop (ATR, after BE)
            if p["use_trail"] and ot.get("be_done"):
                trail_dist = atr_cur * 0.9
                new_sl = c - trail_dist if ot["dir"] == 1 else c + trail_dist
                if ot["dir"] == 1 and new_sl > ot["sl"]:
                    ot["sl"] = new_sl
                if ot["dir"] == -1 and new_sl < ot["sl"]:
                    ot["sl"] = new_sl

            # TP2 hit
            tp2_hit = (ot["dir"]==1 and h >= ot["tp2"]) or (ot["dir"]==-1 and lo <= ot["tp2"])
            sl_hit  = (ot["dir"]==1 and lo <= ot["sl"]) or (ot["dir"]==-1 and h >= ot["sl"])

            if tp2_hit or sl_hit:
                exit_px     = ot["tp2"] if tp2_hit else ot["sl"]
                exit_reason = "TP2" if tp2_hit else "SL"
                pnl = (exit_px - ot["entry"]) * 100000 * ot["lots"] * ot["dir"]
                pnl += ot.get("partial_pnl", 0)
                balance += (exit_px - ot["entry"]) * 100000 * ot["lots"] * ot["dir"]
                ot["exit"]         = exit_px
                ot["pnl"]          = round(pnl, 2)
                ot["exit_reason"]  = exit_reason
                ot["exit_bar"]     = i
                ot["exit_dt"]      = b["dt"]
                trades.append(ot)
                open_trade = None
                if pnl < 0:
                    consec_loss += 1
                else:
                    consec_loss = 0
            continue  # Don't look for new entries while in trade

        # ── Entry filters
        if daily_limit or total_dd_hit:
            continue
        if trades_today >= p["max_trades_day"]:
            continue
        if consec_loss >= p["max_consec_losses"]:
            continue
        # After daily warning: allow max 1 additional trade total
        if daily_warning and extra_trades >= 1:
            continue
        if wday >= 5:
            continue

        # Session filter — London + NY + optional overlap
        in_session = ((p["london"][0]  <= hour < p["london"][1]) or
                      (p["newyork"][0] <= hour < p["newyork"][1]) or
                      (p.get("use_overlap") and p.get("overlap") and
                       p["overlap"][0] <= hour < p["overlap"][1]))
        if not in_session:
            continue

        # Weekend close
        if p["close_weekend"] and wday == 4 and hour >= 20:
            continue

        # News window
        if p["avoid_news"]:
            hhmm = hour * 100 + b["dt"].minute
            news = (1325 <= hhmm <= 1335) or (1555 <= hhmm <= 1605) or (1245 <= hhmm <= 1315)
            if news:
                continue

        # ── Get signals
        smc_sig   = get_smc_signal(bars, i, closes, htf_bull_bias, h4_fast, h4_slow, h4_200, p) if p["use_smc"] else 0
        trend_sig = get_trend_signal(i, closes, vols, ema_f, ema_m, ema_s, rsi_v, atr_v, h4_fast, h4_slow, h4_200, p) if p["use_trend"] else 0
        mr_sig    = get_mr_signal(i, closes, ema_f, ema_s, rsi_v, bb_upper, bb_lower, p) if p["use_mr"] else 0

        # Confluence
        if p["require_confluence"]:
            cmin = p.get("confluence_min", 2)
            bull = sum(1 for s in [smc_sig, trend_sig, mr_sig] if s == 1)
            bear = sum(1 for s in [smc_sig, trend_sig, mr_sig] if s == -1)
            signal = 1 if bull >= cmin else (-1 if bear >= cmin else 0)
        else:
            signal = smc_sig or trend_sig or mr_sig

        if signal == 0:
            continue

        # ── Build trade
        entry   = closes[i]
        atr_cur = atr_v[i]

        if signal == 1:
            recent_low = min(lows[max(0,i-10):i+1])
            sl = max(recent_low - 5e-5, entry - atr_cur * p["atr_sl_mult"])
            sl_dist = entry - sl
        else:
            recent_high = max(highs[max(0,i-10):i+1])
            sl = min(recent_high + 5e-5, entry + atr_cur * p["atr_sl_mult"])
            sl_dist = sl - entry

        if sl_dist <= 0:
            continue

        tp1 = entry + sl_dist * p["tp1_rr"] * signal
        tp2 = entry + sl_dist * p["tp2_rr"] * signal

        # R:R check
        if abs(tp2 - entry) / sl_dist < p["min_rr"]:
            continue

        # Lot sizing — dynamic risk reduction (daily + monthly)
        daily_used = max(0, (day_start_bal - balance) / day_start_bal * 100)
        risk_mult  = 1.0
        if daily_used >= p.get("daily_loss_warning", 1.3):
            risk_mult = 0.5   # halve risk after daily warning
        if consec_loss >= 2:
            risk_mult = min(risk_mult, 0.6)   # reduce after 2 consec losses
        if monthly_halved:
            risk_mult = min(risk_mult, 0.5)   # monthly circuit breaker

        tick_val  = 10.0   # EURUSD: 1 pip = $10 per standard lot
        risk_amt  = balance * p["risk_pct"] / 100 * risk_mult
        lots      = risk_amt / (sl_dist / 0.0001 * tick_val)
        lots      = round(max(0.01, min(50.0, lots)), 2)
        lots      = round(lots / 0.01) * 0.01

        open_trade = {
            "id":        len(trades) + 1,
            "dir":       signal,
            "entry":     entry,
            "sl":        sl,
            "tp1":       tp1,
            "tp2":       tp2,
            "lots":      lots,
            "entry_bar": i,
            "entry_dt":  b["dt"],
            "hour":      hour,
            "unrealised":0,
            "pnl":       0,
            "tp1_done":  False,
            "be_done":   False,
            "smc":       smc_sig,
            "trend":     trend_sig,
            "mr":        mr_sig,
        }
        trades_today += 1
        if daily_warning:
            extra_trades += 1
        htf_bull_bias = (h4_fast.get(i,0) > h4_slow.get(i,0))

    # Close any remaining open trade at last bar
    if open_trade:
        ot = open_trade
        pnl = (closes[-1] - ot["entry"]) * 100000 * ot["lots"] * ot["dir"]
        pnl += ot.get("partial_pnl", 0)
        ot["exit"] = closes[-1]; ot["pnl"] = round(pnl,2)
        ot["exit_reason"] = "END"; ot["exit_dt"] = bars[-1]["dt"]
        trades.append(ot)
        balance += (closes[-1] - ot["entry"]) * 100000 * ot["lots"] * ot["dir"]

    return trades, equity_curve, dd_curve, balance

# ─────────────────────────────────────────────────────────────────────────────
# STATISTICS
# ─────────────────────────────────────────────────────────────────────────────
def compute_stats(trades, equity_curve, start_balance):
    if not trades:
        return {}

    closed = [t for t in trades if "pnl" in t]
    pnls   = [t["pnl"] for t in closed]
    wins   = [p for p in pnls if p > 0]
    losses = [p for p in pnls if p < 0]

    total_pnl    = sum(pnls)
    win_rate     = len(wins) / len(pnls) * 100 if pnls else 0
    avg_win      = np.mean(wins) if wins else 0
    avg_loss     = abs(np.mean(losses)) if losses else 0
    pf           = sum(wins) / abs(sum(losses)) if losses else float("inf")
    max_dd       = max((max(equity_curve[:i+1]) - equity_curve[i]) / max(equity_curve[:i+1]) * 100
                       for i in range(1, len(equity_curve))) if len(equity_curve) > 1 else 0

    eq = np.array(equity_curve)
    returns = np.diff(eq) / eq[:-1]
    sharpe  = (np.mean(returns) / np.std(returns) * np.sqrt(252*96)) if np.std(returns) > 0 else 0

    # Max consecutive losses
    max_consec = cur_c = 0
    for p in pnls:
        if p < 0: cur_c += 1; max_consec = max(max_consec, cur_c)
        else: cur_c = 0

    # Monthly breakdown
    monthly = {}
    for t in closed:
        if "exit_dt" in t:
            key = t["exit_dt"].strftime("%Y-%m")
            monthly[key] = monthly.get(key, 0) + t["pnl"]

    # Exit breakdown
    exit_types = {}
    for t in closed:
        r = t.get("exit_reason","?")
        exit_types[r] = exit_types.get(r, 0) + 1

    return {
        "total_trades":  len(closed),
        "wins":          len(wins),
        "losses":        len(losses),
        "win_rate":      round(win_rate, 1),
        "total_pnl":     round(total_pnl, 2),
        "roi_pct":       round(total_pnl / start_balance * 100, 2),
        "roi_annual":    round(total_pnl / start_balance * 100 / (450/365), 2),
        "avg_win":       round(avg_win, 2),
        "avg_loss":      round(avg_loss, 2),
        "profit_factor": round(pf, 3),
        "max_dd_pct":    round(max_dd, 2),
        "sharpe":        round(sharpe, 3),
        "max_consec_loss": max_consec,
        "monthly":       monthly,
        "exit_types":    exit_types,
    }

# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
print('Engine OK')

In [ ]:
import numpy as np

SS = {}
for _d in range(5):
    for _h in range(24):
        _s = 0.0
        if 8  <= _h <= 10: _s += 0.4
        if 13 <= _h <= 15: _s += 0.4
        if 11 <= _h <= 12: _s += 0.2
        if _d == 0 and _h < 9:   _s *= 0.4
        if _d == 4 and _h >= 15: _s *= 0.3
        if _h < 6 or _h >= 19:  _s = 0.0
        SS[(_d, _h)] = round(min(_s, 1.0), 2)

def h1_rsi_map(bars, period=14):
    h1c=[]; h1i=[]
    for i in range(0, len(bars), 4):
        chunk = bars[i:i+4]
        if chunk: h1c.append(chunk[-1]['close']); h1i.append(i+len(chunk)-1)
    if not h1c: return {}
    h1r = rsi(h1c, period); result = {}
    for idx,bi in enumerate(h1i):
        for j in range(bi-3, bi+1):
            if 0 <= j < len(bars): result[j] = h1r[idx]
    return result

def stepped_trail(ot, c_, hh, ll, atc):
    entry=ot['entry']; sd=abs(entry-ot.get('sl_orig',ot['sl']))
    if sd<=0: return
    d=ot['dir']; prog=((c_-entry)*d)/sd; nsl=ot['sl']
    if prog>=2.5:   cand=c_-atc*0.7*d
    elif prog>=2.0 and not ot.get('t2r'): cand=entry+sd*1.0*d; ot['t2r']=True
    elif prog>=1.0 and not ot.get('be_done'): cand=entry+8e-5*d; ot['be_done']=True
    else: return
    if (d==1 and cand>nsl) or (d==-1 and cand<nsl): ot['sl']=round(cand,5)

print('Miglioramenti v5 OK')


## Parametri — si auto-calibrano per dati reali

In [ ]:
import copy

# Params base
PARAMS = dict(
    use_smc=True, use_trend=True, use_mr=True,
    require_confluence=True, confluence_min=2,
    london=(7,11), newyork=(13,18), overlap=(11,13),
    use_overlap=True, asia=(0,4),
    avoid_news=True, close_weekend=True,
    risk_pct=0.9, max_daily_loss=2.4,
    daily_loss_warning=0.7, max_total_dd=5.5,
    max_trades_day=10, max_consec_losses=3,
    scale_out=True, use_be=True, use_trail=True, be_trigger_rr=1.0,
    tp1_rr=1.5, tp2_rr=3.0, min_rr=1.1, max_spread_pts=28,
    ob_lookback=50, ob_strength=2, ob_body_min=0.00010,
    use_fvg=False, fvg_lookback=20,
    ema_fast=9, ema_med=21, ema_slow=50, ema_200=200,
    rsi_period=14, rsi_long_min=50, rsi_long_max=78,
    rsi_short_min=22, rsi_short_max=50,
    atr_period=14, atr_sl_mult=1.1, use_vol=False,
    bb_period=20, bb_dev=2.0, mr_ob=70, mr_os=30,
    start_balance=10000.0,
)

# Auto-calibra per dati reali (ATR piu' alto)
if REAL_BARS:
    import statistics
    sample = REAL_BARS[:2000]
    atrs = [sample[i]['high']-sample[i]['low'] for i in range(1,len(sample))]
    avg_atr = statistics.mean(atrs)
    bodies  = [abs(b['close']-b['open']) for b in sample]
    avg_body= statistics.mean(bodies)
    print(f'ATR medio reale: {round(avg_atr*10000,1)} pips')
    print(f'Body medio: {round(avg_body*10000,1)} pips')
    # Scala i parametri proporzionalmente
    PARAMS['ob_body_min'] = round(avg_body * 0.8, 5)
    PARAMS['atr_sl_mult'] = 1.5 if avg_atr > 0.0007 else 1.1
    print(f'Params calibrati: ob_body_min={PARAMS["ob_body_min"]:.5f} atr_sl={PARAMS["atr_sl_mult"]}')
else:
    print('Params standard (dati sintetici)')

print('Params OK')


## Esegui Backtest

In [ ]:
import time

def run_v5(bars, p):
    n=len(bars); cl=[b['close']for b in bars]; hi=[b['high']for b in bars]
    lo=[b['low']for b in bars]; vl=[b['volume']for b in bars]
    ef=ema(cl,p['ema_fast']); emv=ema(cl,p['ema_med'])
    es=ema(cl,p['ema_slow']); rv=rsi(cl,p['rsi_period'])
    av=atr(bars,p['atr_period']); bm,bu,bl=bollinger(cl,p['bb_period'],p['bb_dev'])
    h4f,h4s,h4_200=compute_h4_emas(bars,p['ema_fast'],p['ema_slow'],p['ema_200'])
    h1r=h1_rsi_map(bars,p['rsi_period']); ama=ema(av,20)
    bal=p['start_balance']; pk=bal; ds=bal
    pd_=bars[0]['dt'].date(); pm_=bars[0]['dt'].strftime('%Y-%m'); ms=bal; mh=False
    trades=[]; ot=None; tt=0; cl2=0; dl=False; tdd=False; dw=False; et=0; htf=True
    eqc=[]; ddc=[]
    si=max(p['ob_lookback'],p['ema_200'])+10
    for i in range(si,n):
        b=bars[i]; date=b['dt'].date(); h_=b['hour']; wd=b['weekday']
        hhmm=h_*100+b['dt'].minute
        if date!=pd_:
            cm=date.strftime('%Y-%m')
            if cm!=pm_: ms=bal; pm_=cm; mh=False
            if (ms-bal)/ms*100>=2.5 and not mh: mh=True
            ds=bal; tt=0; cl2=0; dl=False; dw=False; et=0; pd_=date
        if ot:
            if ot['dir']==1:
                worst=(lo[i]-ot['entry'])*100000*ot['lots']
                ot['unrealised']=(cl[i]-ot['entry'])*100000*ot['lots']
            else:
                worst=(ot['entry']-hi[i])*100000*ot['lots']
                ot['unrealised']=(ot['entry']-cl[i])*100000*ot['lots']
            eq=bal+ot['unrealised']; weq=bal+worst
        else: eq=weq=bal
        if eq>pk: pk=eq
        ddp=(pk-weq)/pk*100; ddpd=(ds-weq)/ds*100
        eqc.append(eq); ddc.append(ddp)
        if ddp>=p['max_total_dd']:
            if ot: pnl=ot['unrealised']; bal+=pnl; ot.update({'pnl':round(pnl,2),'exit_reason':'TOTAL_DD','exit_dt':b['dt']}); trades.append(ot); ot=None
            tdd=True; break
        if ddpd>=p.get('daily_loss_warning',0.7) and not dw: dw=True
        if ddpd>=p['max_daily_loss']:
            if ot: pnl=ot['unrealised']; bal+=pnl; ot.update({'pnl':round(pnl,2),'exit_reason':'DAILY_DD','exit_dt':b['dt']}); trades.append(ot); ot=None
            dl=True
        if ot:
            c_=cl[i]; hh2=hi[i]; ll2=lo[i]; atc=av[i]
            stepped_trail(ot,c_,hh2,ll2,atc)
            if not ot.get('tp1_done'):
                t1h=(ot['dir']==1 and hh2>=ot['tp1']) or (ot['dir']==-1 and ll2<=ot['tp1'])
                if t1h and p['scale_out']:
                    pp=(ot['tp1']-ot['entry'])*100000*ot['lots']*0.5*ot['dir']
                    bal+=pp; ot['lots']*=0.5; ot['tp1_done']=True
                    ot['partial_pnl']=ot.get('partial_pnl',0)+round(pp,2)
            t2h=(ot['dir']==1 and hh2>=ot['tp2']) or (ot['dir']==-1 and ll2<=ot['tp2'])
            slh=(ot['dir']==1 and ll2<=ot['sl']) or (ot['dir']==-1 and hh2>=ot['sl'])
            if t2h or slh:
                ep=ot['tp2'] if t2h else ot['sl']; er='TP2' if t2h else 'SL'
                pnl=(ep-ot['entry'])*100000*ot['lots']*ot['dir']+ot.get('partial_pnl',0)
                bal+=(ep-ot['entry'])*100000*ot['lots']*ot['dir']
                ot.update({'exit':ep,'pnl':round(pnl,2),'exit_reason':er,'exit_dt':b['dt']}); trades.append(ot); ot=None
                cl2=cl2+1 if pnl<0 else 0
            continue
        if dl or tdd or tt>=p['max_trades_day'] or cl2>=p['max_consec_losses']: continue
        if dw and et>=1: continue
        if wd>=5: continue
        in_s=((p['london'][0]<=h_<p['london'][1]) or (p['newyork'][0]<=h_<p['newyork'][1]) or
              (p.get('use_overlap') and p.get('overlap') and p['overlap'][0]<=h_<p['overlap'][1]))
        if not in_s: continue
        if p.get('close_weekend') and wd==4 and h_>=20: continue
        if p.get('avoid_news'):
            if (1325<=hhmm<=1335) or (1555<=hhmm<=1605) or (1245<=hhmm<=1315): continue
        if i>=20 and ama[i]>0 and av[i]/ama[i]>3.5: continue
        ss=get_smc_signal(bars,i,cl,htf,h4f,h4s,h4_200,p) if p['use_smc'] else 0
        ts_=get_trend_signal(i,cl,vl,ef,emv,es,rv,av,h4f,h4s,h4_200,p) if p['use_trend'] else 0
        ms_=get_mr_signal(i,cl,ef,es,rv,bu,bl,p) if p['use_mr'] else 0
        h1rv=h1r.get(i,50)
        if ts_==1 and h1rv<30: ts_=0
        if ts_==-1 and h1rv>70: ts_=0
        bull=sum(1 for s in [ss,ts_,ms_] if s==1)
        bear=sum(1 for s in [ss,ts_,ms_] if s==-1)
        sig=1 if bull>=p.get('confluence_min',2) else (-1 if bear>=p.get('confluence_min',2) else 0)
        if sig==0: continue
        entry=cl[i]; atc=av[i]
        if sig==1: rl=min(lo[max(0,i-10):i+1]); sl_=max(rl-5e-5,entry-atc*p['atr_sl_mult']); sld=entry-sl_
        else: rh=max(hi[max(0,i-10):i+1]); sl_=min(rh+5e-5,entry+atc*p['atr_sl_mult']); sld=sl_-entry
        if sld<=0: continue
        tp1=entry+sld*p['tp1_rr']*sig; tp2=entry+sld*p['tp2_rr']*sig
        if abs(tp2-entry)/sld<p['min_rr']: continue
        du=max(0,(ds-bal)/ds*100); rm=1.0
        if du>=p.get('daily_loss_warning',0.7): rm=0.5
        if cl2>=2: rm=min(rm,0.6)
        if mh: rm=min(rm,0.5)
        lots=bal*p['risk_pct']/100*rm/(sld/0.0001*10.0)
        lots=round(max(0.01,min(50.0,lots)),2); lots=round(lots/0.01)*0.01
        ot={'id':len(trades)+1,'dir':sig,'entry':entry,'sl':round(sl_,5),'sl_orig':round(sl_,5),
            'tp1':tp1,'tp2':tp2,'lots':lots,'entry_dt':b['dt'],'hour':h_,'weekday':wd,
            'unrealised':0,'pnl':0,'tp1_done':False,'be_done':False,
            'smc':ss,'trend':ts_,'mr':ms_}
        tt+=1
        if dw: et+=1
        htf=(h4f.get(i,0)>h4s.get(i,0))
    if ot:
        pnl=(cl[-1]-ot['entry'])*100000*ot['lots']*ot['dir']+ot.get('partial_pnl',0)
        ot.update({'pnl':round(pnl,2),'exit_reason':'END','exit_dt':bars[-1]['dt']}); trades.append(ot)
        bal+=(cl[-1]-ot['entry'])*100000*ot['lots']*ot['dir']
    return trades,eqc,ddc,bal


# RUN
t0=time.time()
if REAL_BARS:
    BARS=REAL_BARS
    print(f'Backtest su {len(BARS):,} barre REALI Dukascopy...')
else:
    BARS=generate_eurusd_m15(days=450, seed=42)
    print(f'Backtest su {len(BARS):,} barre sintetiche...')

P_run=copy.deepcopy(PARAMS)
trades,eq_curve,dd_curve,final_bal=run_v5(BARS,P_run)
stats=compute_stats(trades,eq_curve,P_run['start_balance'])

daily={}
for t in trades:
    if 'exit_dt' in t:
        d=str(t['exit_dt'])[:10]; daily[d]=daily.get(d,0)+t.get('pnl',0)
max_dl=abs(min(daily.values()))/P_run['start_balance']*100 if daily else 0
monthly={}
for t in trades:
    if 'exit_dt' in t:
        ym=str(t['exit_dt'])[:7]; monthly[ym]=monthly.get(ym,0)+round(t.get('pnl',0),2)

data_type='DATI REALI DUKASCOPY' if REAL_BARS else 'DATI SINTETICI'
print(f'Completato in {time.time()-t0:.1f}s — {data_type}')
print('='*55)
print(f'Trade:          {stats["total_trades"]}')
print(f'Win Rate:       {stats["win_rate"]}%')
print(f'ROI annuo:      +{stats["roi_annual"]}%')
print(f'Profit Factor:  {stats["profit_factor"]}')
print(f'Sharpe:         {stats["sharpe"]}')
print(f'Max Daily Loss: {round(max_dl,2)}%  {"OK" if max_dl<=2.5 else "ATTENZIONE"}')
print(f'Max Total DD:   {stats["max_dd_pct"]}%  {"OK" if stats["max_dd_pct"]<=5.5 else "ATTENZIONE"}')
print(f'Balance finale: EUR{round(final_bal,2)}')
print('='*55)


## Report Visuale

In [ ]:
from IPython.display import HTML, display
import json as J
from datetime import datetime as DT

ms_s=sorted(monthly.items()); ml_s=[m[0]for m in ms_s]; mv_s=[round(m[1],2)for m in ms_s]
pnl_list=[t.get('pnl',0)for t in sorted(trades,key=lambda x:str(x.get('entry_dt','')))]

def hc(v,mx):
    if mx==0: return '#e2e0da'
    r=min(abs(v)/mx,1.0)
    if v>=0: g=int(55+r*145); return f'rgb(10,{g},50)'
    rv=int(100+r*130); return f'rgb({rv},15,25)'
mxM=max(abs(v)for v in mv_s)if mv_s else 1

hm_h=''.join(
    f'<div style="background:{hc(pv,mxM)};border-radius:3px;padding:7px 3px;'
    f'display:flex;flex-direction:column;align-items:center;gap:2px"'
    f'title="{ym}: {chr(43)if pv>=0 else chr(45)}EUR{pv:.0f}">'
    f'<span style="font-family:monospace;font-size:6px;color:rgba(255,255,255,.7)">'
    f'{DT.strptime(ym,"%Y-%m").strftime("%b %y")}</span>'
    f'<span style="font-family:monospace;font-size:9px;color:#fff;font-weight:600">'
    f'{chr(43)if pv>=0 else ""}{int(abs(pv))}</span></div>'
    for ym,pv in ms_s
)

tr_rows=''.join(
    f'<tr style="border-bottom:1px solid #e2e0da">'
    f'<td style="font-family:monospace;font-size:10px;padding:5px 8px 5px 0">#{t.get("id")} </td>'
    f'<td style="font-family:monospace;font-size:10px;padding:5px 8px 5px 0">{str(t.get("entry_dt",""))[:16]}</td>'
    f'<td style="color:{"#0d7a4a"if t.get("dir")==1 else"#c41a3a"};font-family:monospace;font-weight:700;padding:5px 8px 5px 0">{"LONG"if t.get("dir")==1 else"SHORT"}</td>'
    f'<td style="font-family:monospace;font-size:10px;padding:5px 8px 5px 0">{float(t.get("entry",0)):.5f}</td>'
    f'<td style="font-family:monospace;font-size:10px;padding:5px 8px 5px 0;color:#c41a3a">{float(t.get("sl",0)):.5f}</td>'
    f'<td style="font-family:monospace;font-size:10px;padding:5px 8px 5px 0">{t.get("lots",0)}</td>'
    f'<td style="color:{"#0d7a4a"if t.get("pnl",0)>=0 else"#c41a3a"};font-family:monospace;font-weight:700;padding:5px 8px 5px 0">{chr(43)if t.get("pnl",0)>=0 else ""}EUR{float(t.get("pnl",0)):.2f}</td>'
    f'<td style="font-family:monospace;font-size:9px;color:{"#0d7a4a"if"TP"in str(t.get("exit_reason",""))else"#c41a3a"};padding:5px 8px 5px 0">{t.get("exit_reason","?")}</td>'
    f'</tr>'
    for t in sorted(trades,key=lambda x:str(x.get('entry_dt','')))[-40:]
)

is_real=REAL_BARS is not None
dtag='DATI REALI DUKASCOPY' if is_real else 'DATI SINTETICI'
dcol='#0d7a4a' if is_real else '#b85c00'
sb=P_run['start_balance']

h=f"""
<link href='https://fonts.googleapis.com/css2?family=Syne:wght@400;700;800&family=DM+Mono:wght@400;500&display=swap' rel='stylesheet'>
<div style='font-family:Syne,sans-serif;max-width:1050px;margin:0 auto;background:#f5f4f1;padding:14px'>
<div style='background:#0d0d0b;padding:20px 22px;border-radius:4px;margin-bottom:2px;display:flex;justify-content:space-between;align-items:flex-end'>
  <div>
    <div style='margin-bottom:7px'><span style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;padding:3px 8px;border-radius:2px;background:{dcol}33;color:{dcol};border:1px solid {dcol}55'>{dtag}</span></div>
    <div style='font-size:20px;font-weight:800;color:#fff'>RealWinner EA v5</div>
    <div style='font-family:monospace;font-size:8px;color:rgba(255,255,255,.35);letter-spacing:2px;text-transform:uppercase;margin-top:4px'>EURUSD M15 · SMC + Trend + MR</div>
  </div>
  <div style='text-align:right'>
    <div style='font-family:monospace;font-size:46px;font-weight:300;color:#4ade80;letter-spacing:-3px;line-height:1'>+{stats['roi_annual']}%</div>
    <div style='font-family:monospace;font-size:8px;color:rgba(255,255,255,.3);margin-top:3px'>ROI annualizzato · EUR{int(sb):,} > EUR{int(final_bal):,}</div>
  </div>
</div>
<div style='display:grid;grid-template-columns:repeat(4,1fr);gap:2px;margin-bottom:2px'>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>ROI Anno</div><div style='font-family:monospace;font-size:20px;color:#0d7a4a'>+{stats['roi_annual']}%</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Win Rate</div><div style='font-family:monospace;font-size:20px;color:#0d7a4a'>{stats['win_rate']}%</div><div style='font-size:10px;color:#9c9890'>{stats['wins']}W/{stats['losses']}L</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Profit Factor</div><div style='font-family:monospace;font-size:20px;color:#0d7a4a'>{stats['profit_factor']}</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Sharpe</div><div style='font-family:monospace;font-size:20px;color:#1a56db'>{stats['sharpe']}</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Max Daily Loss</div><div style='font-family:monospace;font-size:20px;color:{"#0d7a4a"if max_dl<=2.5 else"#c41a3a"}'>{round(max_dl,2)}%</div><div style='font-size:10px;color:#9c9890'>limite 2.5% {"OK"if max_dl<=2.5 else"ATTENZIONE"}</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Max Total DD</div><div style='font-family:monospace;font-size:20px;color:{"#0d7a4a"if stats['max_dd_pct']<=5.5 else"#c41a3a"}'>{stats['max_dd_pct']}%</div><div style='font-size:10px;color:#9c9890'>limite 5.5% {"OK"if stats['max_dd_pct']<=5.5 else"ATTENZIONE"}</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Avg Win/Loss</div><div style='font-family:monospace;font-size:14px'>EUR{stats['avg_win']}/EUR{stats['avg_loss']}</div></div>
  <div style='background:#fff;padding:12px 14px'><div style='font-family:monospace;font-size:7px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;margin-bottom:4px'>Trade</div><div style='font-family:monospace;font-size:20px'>{stats['total_trades']}</div><div style='font-size:10px;color:#9c9890'>TP2:{stats['exit_types'].get('TP2',0)}</div></div>
</div>
<div style='display:grid;grid-template-columns:3fr 2fr;gap:2px;margin-bottom:2px'>
  <div style='background:#fff;padding:16px 18px'><div style='font-size:11px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:10px'>P&L Mensile</div><canvas id='mCh' height='140'></canvas></div>
  <div style='background:#fff;padding:16px 18px'><div style='font-size:11px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:10px'>Heatmap</div><div style='display:grid;grid-template-columns:repeat(5,1fr);gap:3px'>{hm_h}</div></div>
</div>
<div style='background:#fff;padding:16px 18px;margin-bottom:2px'><div style='font-size:11px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:8px'>P&L per Trade</div><canvas id='pCh' height='75'></canvas></div>
<div style='background:#fff;padding:16px 18px'><div style='font-size:11px;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;margin-bottom:10px'>Trade Log (ultimi 40)</div>
<div style='overflow-y:auto;max-height:250px'><table style='width:100%;border-collapse:collapse'>
<thead><tr style='border-bottom:2px solid #e2e0da'><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>#</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>Data</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>Dir</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>Entry</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>SL</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>Lotti</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>P&L</th><th style='font-family:monospace;font-size:8px;letter-spacing:2px;text-transform:uppercase;color:#9c9890;padding:0 8px 8px 0;text-align:left'>Uscita</th></tr></thead>
<tbody>{tr_rows}</tbody></table></div></div>
</div>
<script src='https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js'></script>
<script>
const ML={J.dumps(ml_s)},MV={J.dumps(mv_s)},PN={J.dumps(pnl_list)};
new Chart(document.getElementById('mCh'),{{type:'bar',data:{{labels:ML,datasets:[{{data:MV,backgroundColor:MV.map(v=>v>=0?'rgba(13,122,74,.75)':'rgba(196,26,58,.75)'),borderColor:MV.map(v=>v>=0?'#0d7a4a':'#c41a3a'),borderWidth:1,borderRadius:3}}]}},options:{{responsive:true,animation:false,plugins:{{legend:{{display:false}}}},scales:{{x:{{grid:{{display:false}},ticks:{{font:{{size:8}},color:'#9c9890',maxRotation:45}}}},y:{{grid:{{color:'rgba(226,224,218,.7)'}},ticks:{{font:{{size:9}},color:'#9c9890',callback:v=>'E'+v}}}}}}}}  }});
new Chart(document.getElementById('pCh'),{{type:'bar',data:{{labels:PN.map((_,i)=>'#'+(i+1)),datasets:[{{data:PN,backgroundColor:PN.map(v=>v>=0?'rgba(13,122,74,.8)':'rgba(196,26,58,.8)'),borderWidth:0,borderRadius:2}}]}},options:{{responsive:true,animation:false,plugins:{{legend:{{display:false}}}},scales:{{x:{{display:false}},y:{{grid:{{color:'rgba(226,224,218,.7)'}},ticks:{{font:{{size:9}},color:'#9c9890',callback:v=>'E'+v}}}}}}}}  }});
</script>
"""

display(HTML(h))
print('Report OK!')


## Scarica Report

In [ ]:
from google.colab import files as _f
with open('RealWinner_Dukascopy_Report.html','w') as f: f.write(h)
_f.download('RealWinner_Dukascopy_Report.html')
print('Scaricato!')
